# Brain Surgery — Bước 0: Vocabulary & PLE Pruning (Gemma 4 E2B)

Notebook chạy trên **Google Colab GPU T4 (15GB VRAM / ~12.7GB CPU RAM)**.

**Luồng thực thi an toàn RAM:**
1. Tải metadata (~30 MB) -> **Preflight Inspection** (`config.json` root + `text_config`, `tokenizer.model`, `ID 5: [multimodal]`, thinking/multimodal tokens).
2. Stream corpus EN/VI/Code -> lọc từ điển 4 tầng + đóng cây BPE Merge-Closure -> căn bội số `128`.
3. Tải shard trọng số -> chạy **Phase 2A** (`capture_orig_logits`) rồi giải phóng sạch RAM/VRAM.
4. Chạy **Low-RAM Shard-by-Shard Slicing** (`< 4-5 GB` peak CPU RAM) + remap đồng bộ `config.json` (root & `text_config`) + xóa dứt điểm `tokenizer.model`.
5. Chạy đủ **4 Cổng Kiểm chứng** (Round-trip + Policy, Logit Equivalence, E2E Generate + Multimodal Towers, Dry-run `llama.cpp` GGUF + `--mmproj`).
6. Đẩy checkpoint đã phẫu thuật + `id_map.json` + `pruning_report.json` lên **Hugging Face Private Repo**.

## 1. Cài đặt môi trường & Nạp module `src/surgery`

In [ ]:
!nvidia-smi
!pip install -q --upgrade "transformers>=5.5.0" "safetensors>=0.4.5" "tokenizers>=0.21.0" "accelerate>=1.0.0" "huggingface_hub>=0.28.0" "datasets>=3.0.0" sentencepiece protobuf

In [ ]:
import gc
import json
import os
from pathlib import Path
import subprocess
import sys

# Chỉnh GITHUB_REPO_URL nếu clone trực tiếp từ GitHub trên Colab;
# hoặc đặt REPO_DIR trỏ tới thư mục đã đồng bộ qua Drive / upload.
GITHUB_REPO_URL = "https://github.com/just-haku/brain-surgery.git"
REPO_DIR = Path("/content/brain_surgery")

if not (REPO_DIR / "src" / "surgery").exists() and GITHUB_REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(REPO_DIR)], check=True)

assert (REPO_DIR / "src" / "surgery").exists(), (
    f"Không tìm thấy {REPO_DIR / 'src' / 'surgery'}. Hãy set GITHUB_REPO_URL hoặc mount Drive/upload thư mục src/."
)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

# Đăng nhập Hugging Face (ưu tiên Colab Secret HF_TOKEN)
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    login()

print("Đã nạp src/surgery và xác thực Hugging Face.")

## 2. Cấu hình Bước 0

In [ ]:
SOURCE_REPO = "treadon/gemma4-E2B-it-abliterated"  # Dự phòng: "Austriani/Gemma-4-E2B-it-ARA-Abliterated"
TARGET_HF_REPO = "haku-nekooo/gemma4-e2b-nuclear-pruned"

TARGET_VOCAB_SIZE = 65536  # Bội số của 128 (49152 hoặc 65536)
VI_FLOOR_QUOTA = 4000
CODE_FLOOR_QUOTA = 16000
CORPUS_MAX_DOCS = {"en": 4000, "vi": 5000, "code": 4500}

WORK_DIR = Path("/content/work_step0")
ORIG_DIR = WORK_DIR / "source_ckpt"
PRUNED_DIR = WORK_DIR / "pruned_ckpt"
CORPUS_CACHE_PATH = WORK_DIR / "corpus_counts.json"
ORIG_LOGITS_PATH = WORK_DIR / "orig_logits_gate2.pt"
GGUF_DIR = WORK_DIR / "gguf_dryrun"
LLAMA_CPP_DIR = Path("/content/llama.cpp")

WORK_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)
print(f"Source: {SOURCE_REPO} -> Target HF Private: {TARGET_HF_REPO} | Target Vocab: {TARGET_VOCAB_SIZE}")

## 3. Preflight Inspection (Kiểm tra thực địa trước khi tải Weights)

Chỉ tải các tệp `.json`, `.jinja`, `tokenizer.model` (~30 MB) để kiểm tra cấu trúc thực tế của `config.json` (root + `text_config`), sự tồn tại của `tokenizer.model`, và toàn bộ special token IDs (`ID 5: [multimodal]`, `<|think|>`, `<|image|>`, `<|audio|>`, `<unused...>`) trước khi chạy cắt.

In [ ]:
from surgery.preflight import download_metadata_only, inspect_checkpoint_metadata

download_metadata_only(SOURCE_REPO, ORIG_DIR, token=HF_TOKEN)
preflight_summary = inspect_checkpoint_metadata(ORIG_DIR, verbose=True)
ORIG_VOCAB_SIZE = preflight_summary["vocab_count"]
assert preflight_summary["vocab_count"] % 128 == 0 or preflight_summary["vocab_count"] == 262144
assert not preflight_summary["missing_processor_files"], (
    f"Thiếu file processor bắt buộc: {preflight_summary['missing_processor_files']}"
)

## 4. Stream Corpus EN + VI + Code & Lọc Từ điển 4 Tầng + BPE Merge-Closure

In [ ]:
from transformers import AutoTokenizer
from surgery.corpus_stats import count_corpus_tokens

orig_tok = AutoTokenizer.from_pretrained(str(ORIG_DIR), use_fast=True)
corpus_counts = count_corpus_tokens(
    orig_tok,
    max_docs_per_lang=CORPUS_MAX_DOCS,
    cache_path=CORPUS_CACHE_PATH,
)
del orig_tok
gc.collect()

for lang, counts in corpus_counts.items():
    print(f"Corpus [{lang}]: {len(counts)} unique token IDs seen")

In [ ]:
from surgery.config_remap import build_id_map
from surgery.vocab_filter import collect_mandatory_whitelist, select_hybrid_vocab

src_config = json.loads((ORIG_DIR / "config.json").read_text(encoding="utf-8"))
src_tok_json = json.loads((ORIG_DIR / "tokenizer.json").read_text(encoding="utf-8"))
src_tok_cfg = (
    json.loads((ORIG_DIR / "tokenizer_config.json").read_text(encoding="utf-8"))
    if (ORIG_DIR / "tokenizer_config.json").exists()
    else None
)
src_gen_cfg = (
    json.loads((ORIG_DIR / "generation_config.json").read_text(encoding="utf-8"))
    if (ORIG_DIR / "generation_config.json").exists()
    else None
)
chat_tpl = (
    (ORIG_DIR / "chat_template.jinja").read_text(encoding="utf-8")
    if (ORIG_DIR / "chat_template.jinja").exists()
    else None
)

vocab_by_id = {int(tid): tok for tok, tid in src_tok_json["model"]["vocab"].items()}
for entry in src_tok_json.get("added_tokens") or []:
    vocab_by_id.setdefault(int(entry["id"]), entry["content"])

whitelist_ids = collect_mandatory_whitelist(
    vocab_by_id=vocab_by_id,
    config=src_config,
    tokenizer_config=src_tok_cfg,
    generation_config=src_gen_cfg,
    tokenizer_json=src_tok_json,
    chat_template=chat_tpl,
)

kept_old_ids, pruned_merges_raw, pruning_report = select_hybrid_vocab(
    vocab_by_id=vocab_by_id,
    raw_merges=src_tok_json["model"]["merges"],
    whitelist_ids=whitelist_ids,
    en_counts=corpus_counts["en"],
    vi_counts=corpus_counts["vi"],
    code_counts=corpus_counts["code"],
    target_size=TARGET_VOCAB_SIZE,
    vi_floor=VI_FLOOR_QUOTA,
    code_floor=CODE_FLOOR_QUOTA,
)
id_map = build_id_map(kept_old_ids)

PRUNED_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "id_map.json").write_text(
    json.dumps({"kept_old_ids": kept_old_ids, "old_to_new": {str(k): v for k, v in id_map.items()}}),
    encoding="utf-8",
)
print(json.dumps(pruning_report, indent=2, ensure_ascii=False))

## 5. Tải Source Shards & Capture Original Logits (Phase 2A cho Cổng 2)

Tải toàn bộ shard trọng số của Source Checkpoint, tính và lưu vector logits của bản gốc tại `kept_old_ids` cho các prompt kiểm thử, rồi giải phóng sạch RAM/VRAM trước khi bước sang cắt shard.

In [ ]:
from huggingface_hub import snapshot_download
from surgery.verify import capture_orig_logits

snapshot_download(repo_id=SOURCE_REPO, local_dir=str(ORIG_DIR), token=HF_TOKEN)
gate2_capture_info = capture_orig_logits(
    orig_dir=ORIG_DIR,
    kept_old_ids=kept_old_ids,
    save_path=ORIG_LOGITS_PATH,
)
print(gate2_capture_info)

## 6. Low-RAM Shard-by-Shard Slicing & Remap Config/Tokenizer

- Đọc tuần tự từng `.safetensors` bằng `safe_open` (cắt theo block hàng `chunk_rows=32768`, không nạp toàn bộ bảng `embed_tokens_per_layer` ~4.7 GB cùng lúc -> đỉnh CPU RAM < 4–5 GB).
- Remap `vocab_size` + toàn bộ special token IDs ở cả **root** lẫn **`text_config`** của `config.json`.
- **Xóa bỏ hoàn toàn `tokenizer.model`** để ép `transformers` và `llama.cpp` chỉ đọc bảng ID mới từ `tokenizer.json`.

In [ ]:
from surgery.config_remap import remap_and_save_metadata
from surgery.shard_slicer import slice_checkpoint_shards

slicing_report = slice_checkpoint_shards(
    src_dir=ORIG_DIR,
    dst_dir=PRUNED_DIR,
    kept_old_ids=kept_old_ids,
    orig_vocab_size=ORIG_VOCAB_SIZE,
)
remap_report = remap_and_save_metadata(
    src_dir=ORIG_DIR,
    dst_dir=PRUNED_DIR,
    kept_old_ids=kept_old_ids,
    pruned_merges_raw=pruned_merges_raw,
    id_map=id_map,
)

full_report = {
    "source_repo": SOURCE_REPO,
    "preflight": preflight_summary,
    "pruning": pruning_report,
    "slicing": slicing_report,
    "remap": remap_report,
}
(PRUNED_DIR / "id_map.json").write_text((WORK_DIR / "id_map.json").read_text(encoding="utf-8"), encoding="utf-8")
(PRUNED_DIR / "pruning_report.json").write_text(json.dumps(full_report, indent=2, ensure_ascii=False), encoding="utf-8")

print("=== SLICING & REMAP SUMMARY ===")
print(f"Peak RSS: {slicing_report['peak_rss_gb']} GB | Saved: {slicing_report['saved_gb']} GB | PLE sliced: {slicing_report['has_ple_sliced']}")
print(f"tokenizer.model action: {remap_report['tokenizer_model_action']}")
print(f"Injected added_tokens: {remap_report['injected_added_tokens']} | Copied aux: {remap_report['copied_auxiliary_files']}")
assert not remap_report["missing_processor_files"], f"Missing processor files in PRUNED_DIR: {remap_report['missing_processor_files']}"

## 7. Cổng Kiểm chứng 1 — Tokenizer Round-Trip & Policy Check

In [ ]:
from surgery.verify import run_gate1_tokenizer

gate1_res = run_gate1_tokenizer(PRUNED_DIR, id_map=id_map)
print(json.dumps(gate1_res, indent=2, ensure_ascii=False))

## 8. Cổng Kiểm chứng 2 & 3 — Logit Equivalence & E2E `Gemma4ForConditionalGeneration`

In [ ]:
from surgery.verify import run_gate2_and_gate3

gate23_res = run_gate2_and_gate3(
    pruned_dir=PRUNED_DIR,
    id_map=id_map,
    orig_logits_path=ORIG_LOGITS_PATH,
)
print(json.dumps(gate23_res, indent=2, ensure_ascii=False))

## 9. Cổng Kiểm chứng 4 — Dry-Run `llama.cpp` GGUF + `--mmproj`

Clone & build bản `llama.cpp` mới nhất hỗ trợ Gemma 4, xuất thử:
- `model-text-code-step0.gguf` (BF16) -> quant `Q4_K_M`.
- `mmproj-vision-audio-step0.gguf` (`--mmproj`).
- Chạy `llama-cli` smoke test và đối chiếu dung lượng với Baseline Brain (`3.45 GB`).

In [ ]:
import subprocess

if not LLAMA_CPP_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP_DIR)], check=True)

!pip install -q --no-deps gguf
!cmake -B {LLAMA_CPP_DIR}/build -S {LLAMA_CPP_DIR} -DGGML_CUDA=OFF
!cmake --build {LLAMA_CPP_DIR}/build --config Release --target llama-quantize llama-cli -j4

In [ ]:
import subprocess

CONVERT_PY = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
QUANTIZE_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"
CLI_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-cli"

main_bf16 = GGUF_DIR / "model-text-code-step0-bf16.gguf"
main_q4 = GGUF_DIR / "model-text-code-step0-Q4_K_M.gguf"
mmproj_bf16 = GGUF_DIR / "mmproj-vision-audio-step0-bf16.gguf"

# 1. Convert main Core Brain GGUF + Quantize Q4_K_M
subprocess.run(
    ["python", str(CONVERT_PY), str(PRUNED_DIR), "--outfile", str(main_bf16), "--outtype", "bf16"],
    check=True,
)
subprocess.run([str(QUANTIZE_BIN), str(main_bf16), str(main_q4), "Q4_K_M"], check=True)

# 2. Convert Multimodal Add-on (--mmproj)
mmproj_ok = False
mmproj_proc = subprocess.run(
    ["python", str(CONVERT_PY), str(PRUNED_DIR), "--mmproj", "--outfile", str(mmproj_bf16), "--outtype", "bf16"],
    capture_output=True,
    text=True,
)
if mmproj_proc.returncode == 0:
    mmproj_ok = True
    print(f"--mmproj PASSED: {mmproj_bf16.name} ({mmproj_bf16.stat().st_size / (1024**3):.3f} GB)")
else:
    print("--mmproj FAILED (kiểm tra hỗ trợ audio/vision Gemma 4 trong phiên bản llama.cpp hiện tại):")
    print(mmproj_proc.stderr[-1500:])

# 3. Smoke test suy luận bằng llama-cli
cli_proc = subprocess.run(
    [
        str(CLI_BIN),
        "-m", str(main_q4),
        "-p", "Viết hàm Python kiểm tra số nguyên tố:",
        "-n", "48",
        "--no-display-prompt",
    ],
    capture_output=True,
    text=True,
    check=True,
)

q4_gb = main_q4.stat().st_size / (1024**3)
print(f"Core Brain Q4_K_M size: {q4_gb:.3f} GB (so với Baseline Brain Gemma-4-E2B-IT-AB.gguf: 3.213 GiB / 3.45 GB)")
print("llama-cli preview:\n", cli_proc.stdout[:400])

full_report["gates"] = {
    "gate1": gate1_res,
    "gate23": gate23_res,
    "gate4": {
        "main_q4_km_gb": round(q4_gb, 3),
        "mmproj_exported": mmproj_ok,
        "mmproj_gb": round(mmproj_bf16.stat().st_size / (1024**3), 3) if mmproj_ok else None,
    },
}
(PRUNED_DIR / "pruning_report.json").write_text(json.dumps(full_report, indent=2, ensure_ascii=False), encoding="utf-8")
assert mmproj_ok, "Gate 4 warning: main GGUF passed nhưng --mmproj gặp lỗi (xem log phía trên)."

## 10. Đẩy Pruned Checkpoint lên Hugging Face Private Repo

In [ ]:
from huggingface_hub import HfApi

assert "your-username" not in TARGET_HF_REPO, "Hãy đổi TARGET_HF_REPO thành repo Hugging Face của bạn trước khi upload."

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=TARGET_HF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_folder(
    folder_path=str(PRUNED_DIR),
    repo_id=TARGET_HF_REPO,
    repo_type="model",
    commit_message=f"Step 0: pruned Gemma 4 E2B vocab to {TARGET_VOCAB_SIZE} + passed 4 gates",
)
print(f"Hoàn tất đẩy Bước 0 checkpoint lên HF Private Repo: https://huggingface.co/{TARGET_HF_REPO}")